# Evaluate NuTime with SVM on UEA Multivariate Datasets

NuTime checkpoint is pretrained univariate (Conv1d(1,128,16), channel_embedding=Identity).
We process each channel independently: (N,C,L) -> (N*C,1,L) -> features -> (N, C*128) -> SVM.

In [ ]:
import os
import sys
import json
import torch
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '.')))
from config import Config
from models.build import get_model
from sktime.datasets import load_from_tsfile

In [ ]:
import models.networks as net

def _patched_forward(self, x):
    import torch
    B, C, L = x.shape

    if self.window_slide:
        x = x.view(-1, L).unsqueeze(1)                          # (B*C, 1, L)
        x_conv = self.window_embedding(x)                        # (B*C, embed_dim, N_actual)
        N = x_conv.shape[2]                                      # dynamic, not self.num_windows
        x_embed = x_conv.transpose(1, 2).contiguous().view(B, C, N, -1)
        x_embed = self.window_embed_norm(x_embed)
        x_embed = x_embed.transpose(1, 2).contiguous().view(B, N, -1)
        x_embed = self.channel_embedding(x_embed)
    else:
        x_embed = x.transpose(1, 2)
        N = x_embed.shape[1]

    if self.transformer_mask_type == 'learnable':
        mask_tokens = int(self.mask_token_num * N / max(self.num_windows, 1))
        bool_masked_pos = self.generate_block_mask(num_tokens=N, mask_tokens=mask_tokens)
        mask_token = self.mask_token.expand(B, N, -1)
        w = bool_masked_pos.unsqueeze(-1).type_as(mask_token)
        x_embed = x_embed * (1 - w) + mask_token * w

    if self.cls_token is not None:
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x_embed = torch.cat((cls_tokens, x_embed), dim=1)

    x_embed += self.pos_embed[:, :x_embed.shape[1], :]
    x_embed = self.pos_drop(x_embed)

    if self.transformer_mask_type == 'drop':
        mask = self.generate_random_mask(num_tokens=N, mask_tokens=self.mask_token_num)
        x_embed = x_embed[:, ~mask].reshape(B, -1, x_embed.shape[-1])

    output, attn = self.transformer(x_embed)
    output_norm = self.norm(output)
    self.last_attn = attn
    self.features = output_norm[:, 0]
    return self.fc(self.features)

net.WinT.forward = _patched_forward
print("Patched WinT.forward: dynamic N + contiguous fix")

In [ ]:
def _pad_nested(df):
    max_len = 0
    for _, row in df.iterrows():
        for s in row:
            if len(s) > max_len:
                max_len = len(s)
    N, C = df.shape
    out = np.zeros((N, C, max_len), dtype=np.float32)
    for i, (_, row) in enumerate(df.iterrows()):
        for c, s in enumerate(row):
            v = s.to_numpy(dtype=np.float32)
            v = np.nan_to_num(v, nan=0.0)
            out[i, c, :len(v)] = v
    return out


def load_uea_ts(data_dir, dataset_name):
    train_file = os.path.join(data_dir, dataset_name, f"{dataset_name}_TRAIN.ts")
    test_file = os.path.join(data_dir, dataset_name, f"{dataset_name}_TEST.ts")
    if not os.path.exists(train_file) or not os.path.exists(test_file):
        return None, None, None, None
    try:
        X_train, y_train = load_from_tsfile(train_file, return_data_type="numpy3d")
        X_test, y_test = load_from_tsfile(test_file, return_data_type="numpy3d")
    except Exception:
        X_train_nested, y_train = load_from_tsfile(train_file, return_data_type="nested_univ")
        X_test_nested, y_test = load_from_tsfile(test_file, return_data_type="nested_univ")
        train_arr = _pad_nested(X_train_nested)
        test_arr = _pad_nested(X_test_nested)
        L = max(train_arr.shape[2], test_arr.shape[2])
        if train_arr.shape[2] < L:
            pad = np.zeros((train_arr.shape[0], train_arr.shape[1], L - train_arr.shape[2]), dtype=np.float32)
            train_arr = np.concatenate([train_arr, pad], axis=2)
        if test_arr.shape[2] < L:
            pad = np.zeros((test_arr.shape[0], test_arr.shape[1], L - test_arr.shape[2]), dtype=np.float32)
            test_arr = np.concatenate([test_arr, pad], axis=2)
        X_train, X_test = train_arr, test_arr
    return X_train.astype(np.float32), y_train, X_test.astype(np.float32), y_test


def maybe_downsample(X, max_len):
    seq_len = X.shape[2]
    if seq_len <= max_len:
        return X, 1
    stride = int(np.ceil(seq_len / max_len))
    return X[:, :, ::stride], stride

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import json

# ═══════════════════════════════════════════════════════════
# Seed + Random Init helpers (same as Mantis / MOMENT / UniTS)
# ═══════════════════════════════════════════════════════════

def set_seed(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def reset_weights(m):
    if isinstance(m, (nn.Linear, nn.Conv1d, nn.Conv2d)):
        nn.init.xavier_uniform_(m.weight.data)
        if m.bias is not None:
            m.bias.data.zero_()
    elif isinstance(m, nn.LayerNorm):
        m.weight.data.fill_(1.0)
        m.bias.data.zero_()
    elif isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d)):
        m.weight.data.fill_(1.0)
        m.bias.data.zero_()
        m.running_mean.zero_()
        m.running_var.fill_(1.0)
    elif isinstance(m, nn.Embedding):
        nn.init.xavier_uniform_(m.weight.data)


# ═══════════════════════════════════════════════════════════
# Config
# ═══════════════════════════════════════════════════════════

INIT = "pretrained"     # "pretrained" or "random"
SEED = 0
checkpoint_path = "ckpt/checkpoint_bias9.pth"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# --- Config from pretraining ---
default_config = json.load(open("configs/default_ssl.json"))
config = Config()
config.update_by_dict(default_config)
config.model = 'wint'
config.task = 'cls'

# Same architecture config as before (must match pretraining to keep apples-to-apples)
config.encoder = 'cnn'
config.num_channels         = 1
config.window_emb_dim       = 128
config.window_size          = 16
config.stride               = 16
config.transformer_depth    = 6
config.transformer_heads    = 8
config.transformer_head_dim = 64
config.transformer_mlp_dim  = 512
config.transformer_max_tokens = 1024
config.model_series_size    = 1024
config.num_classes          = 2
config.out_dim              = 2
config.transformer_mask_type  = 'none'
config.transformer_mask_scale = 0.0

print("Architecture config:")
for k in ['encoder', 'num_channels', 'window_emb_dim', 'window_size', 'stride',
          'transformer_depth', 'transformer_heads', 'transformer_head_dim',
          'transformer_mlp_dim', 'transformer_max_tokens', 'model_series_size',
          'transformer_mask_type']:
    print(f"  {k}: {getattr(config, k, '<MISSING>')}")

assert config.window_emb_dim % config.transformer_heads == 0

# ═══════════════════════════════════════════════════════════
# Build model + init
# ═══════════════════════════════════════════════════════════

# Set seed BEFORE building the model so random init is reproducible
set_seed(SEED)

print(f"\nBuilding NuTime ({INIT} init, seed={SEED})...")
model = get_model(config)

if INIT == "pretrained":
    # --- Load checkpoint and strip BYOL wrapper ---
    raw_state = torch.load(checkpoint_path, map_location='cpu')
    raw_sd = raw_state['state_dict'] if (isinstance(raw_state, dict) and 'state_dict' in raw_state) else raw_state

    cleaned_state = {}
    for k, v in raw_sd.items():
        if k.startswith('momentum_') or k.startswith('predictor.'):
            continue
        nk = k.replace('module.', '').replace('backbone.', '').replace('online_network.', '')
        if nk.startswith('0.') or nk.startswith('1.'):
            nk = nk[2:]
        if nk.startswith('fc.'):
            continue
        cleaned_state[nk] = v

    print(f"State dict: {len(raw_sd)} raw -> {len(cleaned_state)} cleaned")

    model_keys = set(model.state_dict().keys())
    ckpt_keys = set(cleaned_state.keys())
    print(f"Model keys: {len(model_keys)} | Checkpoint keys: {len(ckpt_keys)}")
    print(f"In model but not checkpoint: {model_keys - ckpt_keys}")
    print(f"In checkpoint but not model: {ckpt_keys - model_keys}")

    load_result = model.load_state_dict(cleaned_state, strict=False)
    print(f"missing={len(load_result.missing_keys)} | unexpected={len(load_result.unexpected_keys)}")
    if load_result.missing_keys:
        print(f"  missing: {load_result.missing_keys}")
    if load_result.unexpected_keys:
        print(f"  unexpected: {load_result.unexpected_keys}")

elif INIT == "random":
    # Re-seed right before reset so the same seed -> same random weights
    set_seed(SEED)
    model.apply(reset_weights)
    print(f"  Applied reset_weights (Xavier init, seed={SEED}) — no checkpoint loaded")

else:
    raise ValueError(f"Unknown init: {INIT}")

model.to(device)
model.eval()
print("\nModel ready.")

In [ ]:
def extract_features_univariate(model, X, batch_size=64, aggregation='concat'):
    """
    Process each channel independently through univariate NuTime backbone.
    X: (N, C, L) -> features: (N, C*feat_dim) or (N, feat_dim)
    """
    N, C, L = X.shape
    # Each (sample, channel) pair becomes its own univariate series
    X_flat = X.reshape(N * C, 1, L)  # (N*C, 1, L)

    feats_list = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(X_flat), batch_size):
            batch = torch.tensor(X_flat[i:i + batch_size], dtype=torch.float32).to(device)

            # Instance normalization per univariate series
            mean = batch.mean(dim=2, keepdim=True)
            std = batch.std(dim=2, keepdim=True)
            batch = (batch - mean) / (std + 1e-8)

            _ = model(batch)
            feats_list.append(model.features.cpu().numpy())

    feats = np.concatenate(feats_list, axis=0)  # (N*C, feat_dim)
    feat_dim = feats.shape[1]
    feats = feats.reshape(N, C, feat_dim)        # (N, C, feat_dim)

    if aggregation == 'concat':
        return feats.reshape(N, C * feat_dim)    # (N, C*feat_dim)
    elif aggregation == 'mean':
        return feats.mean(axis=1)                # (N, feat_dim)
    else:
        raise ValueError(f"Unknown aggregation: {aggregation}")

In [ ]:
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression, RidgeClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score


base_data_dir = "dataset"
datasets = sorted(
    d for d in os.listdir(base_data_dir)
    if os.path.isdir(os.path.join(base_data_dir, d))
)

AGGREGATION = 'concat'
HIGH_CHANNEL_THRESHOLD = 64  # auto-switch to mean for very high channel count

# Max input length NuTime can handle = window_size * max_tokens
MAX_INPUT_LEN = config.window_size * config.transformer_max_tokens  # 16 * 1024 = 16384


# ═══════════════════════════════════════════════════════════
# Classifiers (same set used across Mantis / MOMENT / UniTS)
# ═══════════════════════════════════════════════════════════

def get_classifiers():
    classifiers = {}

    # SVM (RBF) — GridSearch over C
    classifiers["SVM"] = {
        "pipeline": Pipeline([
            ("scaler", StandardScaler()),
            ("svm", SVC(kernel="rbf", random_state=42)),
        ]),
        "param_grid": {"svm__C": [0.01, 0.1, 1.0, 10.0, 100.0]},
        "use_gridsearch": True,
    }

    # Logistic Regression — GridSearch over C
    classifiers["LogReg"] = {
        "pipeline": Pipeline([
            ("scaler", StandardScaler()),
            ("logreg", LogisticRegression(max_iter=2000, random_state=42)),
        ]),
        "param_grid": {"logreg__C": [0.01, 0.1, 1.0, 10.0, 100.0]},
        "use_gridsearch": True,
    }

    # RidgeClassifierCV — built-in CV over alphas
    classifiers["Ridge"] = {
        "pipeline": Pipeline([
            ("scaler", StandardScaler()),
            ("ridge", RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))),
        ]),
        "param_grid": None,
        "use_gridsearch": False,
    }

    # Random Forest
    classifiers["RF"] = {
        "pipeline": Pipeline([
            ("scaler", StandardScaler()),
            ("rf", RandomForestClassifier(n_estimators=200, random_state=42)),
        ]),
        "param_grid": None,
        "use_gridsearch": False,
    }

    return classifiers


def fit_and_score(train_features, y_train, test_features, y_test, n_splits):
    """Fit all classifiers, return dict {clf_name: {'Acc':..., 'MF1':..., 'best_params':...}}."""
    classifiers = get_classifiers()
    out = {}
    for clf_name, cfg in classifiers.items():
        try:
            if cfg["use_gridsearch"] and n_splits >= 2:
                gs = GridSearchCV(
                    cfg["pipeline"],
                    cfg["param_grid"],
                    cv=n_splits,
                    scoring="accuracy",
                    n_jobs=-1,
                    refit=True,
                )
                gs.fit(train_features, y_train)
                y_pred = gs.predict(test_features)
                best = gs.best_params_
            else:
                # Either no GridSearch needed, or too few samples for CV → fit pipeline directly
                pipe = cfg["pipeline"]
                pipe.fit(train_features, y_train)
                y_pred = pipe.predict(test_features)
                best = None

            acc = accuracy_score(y_test, y_pred)
            mf1 = f1_score(y_test, y_pred, average="macro")
            out[clf_name] = {"Acc": acc, "MF1": mf1, "best_params": best}
            best_str = f" | best: {best}" if best is not None else ""
            print(f"    {clf_name:7s} Acc={acc:.4f} MF1={mf1:.4f}{best_str}")
        except Exception as e:
            print(f"    {clf_name:7s} FAILED — {type(e).__name__}: {e}")
            out[clf_name] = {"Acc": np.nan, "MF1": np.nan, "best_params": None}

    return out


# ═══════════════════════════════════════════════════════════
# Main loop
# ═══════════════════════════════════════════════════════════

results = {}
print(f"Found {len(datasets)} datasets.\n")

for dataset_name in datasets:
    print(f"Processing {dataset_name}...")
    try:
        X_train, y_train, X_test, y_test = load_uea_ts(base_data_dir, dataset_name)
        if X_train is None:
            print("  -> Skipping, missing TRAIN/TEST files.\n")
            continue

        N_train, channel, seq_len = X_train.shape
        num_classes = len(np.unique(y_train))
        print(f"  train: {X_train.shape}, test: {X_test.shape}, classes: {num_classes}")

        # Downsample if sequence too long for positional embedding
        X_train, stride = maybe_downsample(X_train, MAX_INPUT_LEN)
        X_test, _ = maybe_downsample(X_test, MAX_INPUT_LEN)
        if stride > 1:
            print(f"  -> Downsampled with stride {stride} to length {X_train.shape[2]}")

        # Aggregation strategy
        agg = AGGREGATION
        if channel > HIGH_CHANNEL_THRESHOLD and agg == 'concat':
            agg = 'mean'
            print(f"  -> High channel count ({channel}), using mean aggregation")

        # Extract features
        train_features = extract_features_univariate(model, X_train, aggregation=agg)
        test_features = extract_features_univariate(model, X_test, aggregation=agg)
        print(f"  features: train={train_features.shape}, test={test_features.shape}")

        # CV split count (respect smallest class)
        min_class_count = int(np.min(np.unique(y_train, return_counts=True)[1]))
        if N_train < 5 or min_class_count < 2:
            print(f"  -> Too small for CV (N={N_train}, min_class={min_class_count}); "
                  "fitting without GridSearchCV.")
            n_splits = 1  # signals fit_and_score to skip GridSearch
        else:
            n_splits = max(2, min(5, min_class_count))

        # Fit & score all classifiers
        clf_results = fit_and_score(
            train_features, y_train, test_features, y_test, n_splits
        )

        results[dataset_name] = {
            "channels": channel,
            "seq_len": X_train.shape[2],
            "agg": agg,
            "classifiers": clf_results,
        }
        print()

    except Exception as e:
        import traceback
        print(f"  [Error] {type(e).__name__}: {e}")
        traceback.print_exc()
        print()


# ═══════════════════════════════════════════════════════════
# Summary
# ═══════════════════════════════════════════════════════════

print("=" * 70)
print("Final Results:")
print("=" * 70)

clf_names = ["SVM", "LogReg", "Ridge", "RF"]

# Header
header = f"{'Dataset':<28s} " + " ".join(f"{name:>10s}" for name in clf_names)
print(header)
print("-" * len(header))

for ds_name, ds_res in results.items():
    accs = [ds_res["classifiers"].get(c, {}).get("Acc", np.nan) for c in clf_names]
    row = f"{ds_name:<28s} " + " ".join(
        f"{a:>10.4f}" if not np.isnan(a) else f"{'NaN':>10s}" for a in accs
    )
    print(row)

# Mean across datasets per classifier
if results:
    print("-" * len(header))
    means = []
    for c in clf_names:
        col = [ds_res["classifiers"].get(c, {}).get("Acc", np.nan)
               for ds_res in results.values()]
        col = [x for x in col if not np.isnan(x)]
        means.append(np.mean(col) if col else np.nan)
    mean_row = f"{'MEAN':<28s} " + " ".join(
        f"{m:>10.4f}" if not np.isnan(m) else f"{'NaN':>10s}" for m in means
    )
    print(mean_row)

# Optional: also save to CSV in the same layout the other scripts use
try:
    import pandas as pd
    rows = {}
    for ds_name, ds_res in results.items():
        rows[ds_name] = {c: ds_res["classifiers"].get(c, {}).get("Acc", np.nan)
                         for c in clf_names}
    df = pd.DataFrame(rows).T
    df.index.name = "Dataset"
    df.loc["MEAN"] = df.mean()
    out_csv = "nutime_results.csv"
    df.to_csv(out_csv, float_format="%.4f")
    print(f"\nSaved to {out_csv}")
except Exception as e:
    print(f"(CSV save skipped: {e})")

In [ ]:
if results:
    df = pd.DataFrame(results).T
    df.index.name = 'dataset'
    df = df.reset_index()
    out_path = 'nutime_svm_seed_7.csv'
    df.to_csv(out_path, index=False)
    print(f"Saved to {out_path}")
    print(f"Mean Acc: {df['Acc'].mean():.4f} | Mean MF1: {df['MF1'].mean():.4f}")